# Импорт библиотек

In [2]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(1001, 213)

# Удаление невалидных экземляров

In [6]:
matching_instances = df[df['IC50, mM'] == df['CC50, mM']]
print(len(matching_instances.index))
df.drop(index=matching_instances.index, inplace=True)
df.shape

135


(866, 213)

# Очистка таргета от выбросов

In [8]:
df = df[df['SI'] < 500]
df.shape

(849, 213)

# Подготовка данных для эксперемента

In [10]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

In [11]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['SI'].apply(lambda v: 1 if v >= df['SI'].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (679, 98), (679,)
Test dataset size: (170, 98), (170,)


# Эксперемент с моделями

In [13]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
4,XGBoost,0.72,0.72,0.72,0.72,170.0,0.78
7,CatBoost,0.72,0.72,0.72,0.72,170.0,0.78
6,LightGBM,0.71,0.71,0.71,0.71,170.0,0.79
9,AdaBoost,0.69,0.69,0.69,0.69,170.0,0.79
5,Gradient Boosting,0.68,0.68,0.68,0.68,170.0,0.77
1,Decision Tree,0.67,0.67,0.67,0.67,170.0,0.70
3,Random Forest,0.67,0.67,0.67,0.67,170.0,0.77
8,HistGradientBoosting,0.67,0.67,0.67,0.67,170.0,0.77
2,KNeighbors,0.66,0.66,0.69,0.66,170.0,0.70
0,Logistic Regression,0.65,0.65,0.66,0.65,170.0,0.69


# Подбор гиперпараметров

In [31]:
param_dist = {
    'n_estimators': randint(50, 300),
    'num_leaves': randint(20, 50),
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
    'min_child_samples': randint(5, 20),
    'min_split_gain': uniform(0, 0.1),
}
random_search = RandomizedSearchCV(
    estimator=LGBMClassifier(verbose=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    verbose=0,
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

Лучшие параметры: {'colsample_bytree': 0.7874772639179881, 'learning_rate': 0.13444585070129955, 'max_depth': 3, 'min_child_samples': 9, 'min_split_gain': 0.005637549665092712, 'n_estimators': 55, 'num_leaves': 47, 'reg_alpha': 0.8129010091300776, 'reg_lambda': 0.9997176732861306, 'subsample': 0.9986547348295621}


In [35]:
model = LGBMClassifier(
    verbose=-1,
    colsample_bytree=0.79,
    learning_rate=0.134,
    max_depth=3,
    min_child_samples=9,
    min_split_gain=0.006,
    n_estimators=55,
    num_leaves=47,
    reg_alpha=0.81,
    reg_lambda=1,
    subsample=1,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.69
ROC AUC Score: 0.77


,precision,recall,f1-score,support
0,0.686047,0.702381,0.694118,84.000000
1,0.702381,0.686047,0.694118,86.000000
accuracy,0.694118,0.694118,0.694118,0.694118
macro avg,0.694214,0.694214,0.694118,170.000000
weighted avg,0.694310,0.694118,0.694118,170.000000
